<a href="https://colab.research.google.com/github/suyogbastakoti/Ai-and-Ml-Module/blob/main/2407093_SuyogBastakoti_2025_W08_Text_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trump Tweet Sentiment Classification
In this notebook, we will:
1. Clean and preprocess tweet text
2. Convert text to numbers using TF-IDF
3. Train a machine learning model to classify sentiment
4. Evaluate how well our model performs

## Step 0: Mount Google Drive
This lets Colab access files stored in your Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1: Install & Import Libraries
We need a few libraries. Most come pre-installed in Colab, but we install `nltk` data just in case.

In [2]:
# ── Installing NLTK data ──
import nltk
nltk.download('stopwords')      # Common words to remove ("the", "is", "at" ...)
nltk.download('wordnet')        # Dictionary used for lemmatization
nltk.download('omw-1.4')        # Extra language data for WordNet

# ── Standard libraries ───────────────────────────────────────────────────────
import re                        # Regular expressions – for pattern-based text cleaning
import pandas as pd              # For loading and handling our dataset

# ── NLTK tools ───────────────────────────────────────────────────────────────
from nltk.corpus import stopwords          # List of common English stopwords
from nltk.stem import WordNetLemmatizer    # Converts words to their base form (e.g. "running" → "run")
from nltk.stem import PorterStemmer        # Strips word suffixes (e.g. "running" → "run")

# ── Scikit-learn tools ───────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split      # Split data into train/test
from sklearn.feature_extraction.text import TfidfVectorizer  # Convert text → numbers
from sklearn.linear_model import LogisticRegression       # Our ML model
from sklearn.metrics import classification_report         # Evaluate model performance

print("All libraries imported successfully!")

All libraries imported successfully!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


## Step 2: Build the Text Cleaning Pipeline

Raw tweets contain a lot of noise — URLs, emojis, hashtag symbols, punctuation, etc.
This function cleans all of that up so the model only sees meaningful words.

**What each step does:**
| Step | Example |
|---|---|
| Lowercase | `"Trump WINS"` → `"trump wins"` |
| Remove URLs | `"check https://t.co/abc"` → `"check"` |
| Remove emojis | `"great 😀"` → `"great"` |
| Remove special chars | `"#MAGA!!"` → `"maga"` |
| Remove stopwords | `"he is the best"` → `"best"` |
| Lemmatize | `"running"` → `"run"` |

In [3]:
# Initialise tools we'll use inside the function
lemmatizer = WordNetLemmatizer()
stemmer    = PorterStemmer()
stop_words = set(stopwords.words('english'))  # Set is faster for lookup


def text_cleaning_pipeline(text, rule="lemmatize"):
    """
    Cleans a single tweet string through several preprocessing steps.

    Parameters
    ----------
    text : str
        The raw tweet text.
    rule : str
        Either 'lemmatize' (default) or 'stem'.

    Returns
    -------
    str
        A cleaned, space-joined string of tokens.
    """
    # 1. Convert to lowercase so "Trump" and "trump" are treated the same
    data = text.lower()

    # 2. Remove URLs  (anything starting with http / https / www)
    data = re.sub(r'http\S+|www\S+|https\S+', '', data, flags=re.MULTILINE)

    # 3. Remove emojis and other non-ASCII characters
    data = data.encode('ascii', 'ignore').decode('ascii')

    # 4. Remove everything that is NOT a letter or a space
    #    This gets rid of @mentions, #hashtag symbols, punctuation, numbers …
    data = re.sub(r'[^a-z\s]', '', data)

    # 5. Tokenise – split the cleaned string into individual words
    tokens = data.split()

    # 6. Remove stopwords  (e.g. "the", "is", "at", "which", "on" …)
    tokens = [word for word in tokens if word not in stop_words]

    # 7. Lemmatize OR Stem, depending on the 'rule' argument
    if rule == "lemmatize":
        # Lemmatization keeps real dictionary words  ("better" → "good")
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    elif rule == "stem":
        # Stemming is faster but may produce non-words  ("studies" → "studi")
        tokens = [stemmer.stem(word) for word in tokens]
    else:
        print("Pick between 'lemmatize' or 'stem'")

    # Join tokens back into a single string and return
    return " ".join(tokens)


# ── Quick sanity check ───────────────────────────────────────────────────────
sample = "RT @JohnDoe: Trump is AMAZING!!! 🔥 Check this out https://t.co/abc123 #MAGA"
print("Original :", sample)
print("Cleaned  :", text_cleaning_pipeline(sample))

Original : RT @JohnDoe: Trump is AMAZING!!! 🔥 Check this out https://t.co/abc123 #MAGA
Cleaned  : rt johndoe trump amazing check maga


## Step 3: Load the Dataset

Upload `trum_tweet_sentiment_analysis.csv` to your Google Drive and update the path below.

> **Alternatively**, if you upload the file directly to Colab (Files panel on the left), use:
> `pd.read_csv('trum_tweet_sentiment_analysis.csv')`

In [4]:
# ── Load from Google Drive ──

df = pd.read_csv('/content/drive/MyDrive/Data Set /trum_tweet_sentiment_analysis.csv')

# ── Preview ──
print(f"Dataset shape: {df.shape}  →  {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst 5 rows:")
df.head()

Dataset shape: (1850123, 2)  →  1850123 rows, 2 columns

First 5 rows:


,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [5]:
# ── Check label distribution ─────────────────────────────────────────────────
print("Label counts:")
print(df['Sentiment'].value_counts())
print("\nLabel meaning:  0 = Negative/Neutral   |   1 = Positive")

Label counts:
Sentiment
0    1244211
1     605912
Name: count, dtype: int64

Label meaning:  0 = Negative/Neutral   |   1 = Positive


## Step 4: Apply Text Cleaning

We now run every tweet through our cleaning pipeline.
This can take a minute because there are thousands of tweets — a progress message will appear.

In [6]:
# Drop any rows where the text column is empty / NaN
df = df.dropna(subset=['text'])

print("Cleaning tweets … this may take 30–60 seconds …")

# Apply the pipeline to every row in the 'text' column
df['cleaned_text'] = df['text'].apply(text_cleaning_pipeline)

print("Done!")
print("\nSample comparison:")
print("ORIGINAL :", df['text'].iloc[0])
print("CLEANED  :", df['cleaned_text'].iloc[0])

Cleaning tweets … this may take 30–60 seconds …
Done!

Sample comparison:
ORIGINAL : RT @JohnLeguizamo: #trump not draining swamp but our taxpayer dollars on his trips to advertise his properties! @realDonaldTrump https://t.co/gFBvUkMX9z
CLEANED  : rt johnleguizamo trump draining swamp taxpayer dollar trip advertise property realdonaldtrump


## Step 5: Train / Test Split

We split the data so the model trains on 80% and we test on the remaining 20%.
The model **never sees** the test set during training — this gives us an honest evaluation.

`random_state=42` just means the split is reproducible (we'll get the same split every run).

In [7]:
X = df['cleaned_text']   # Features  → the cleaned tweet text
y = df['Sentiment']      # Labels    → 0 or 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% goes to testing
    random_state=42      # Fixes the random seed for reproducibility
)

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")

Training samples : 1480098
Testing  samples : 370025


## Step 6: TF-IDF Vectorization

Machine learning models can't work with raw text — they need numbers.
**TF-IDF** (Term Frequency–Inverse Document Frequency) converts each tweet into a vector of numbers that represents how important each word is.

> `max_features=5000` means we keep only the top 5000 most useful words to keep things fast.

In [8]:
# Create the vectorizer
vectorizer = TfidfVectorizer(
    max_features=5000,   # Only keep the 5000 most informative words
    ngram_range=(1, 2)   # Consider single words AND two-word phrases (bigrams)
)

# fit_transform on TRAINING data: learns the vocabulary + converts to numbers
X_train_tfidf = vectorizer.fit_transform(X_train)

# transform on TEST data: uses the SAME vocabulary (no peeking!)
X_test_tfidf  = vectorizer.transform(X_test)

print(f"TF-IDF matrix shape (train): {X_train_tfidf.shape}")
print(f"TF-IDF matrix shape (test) : {X_test_tfidf.shape}")
print("\nEach row = one tweet, each column = one word/phrase from the vocabulary.")

TF-IDF matrix shape (train): (1480098, 5000)
TF-IDF matrix shape (test) : (370025, 5000)

Each row = one tweet, each column = one word/phrase from the vocabulary.


## Step 7: Train the Model

We use **Logistic Regression** — a simple but powerful model for text classification.
It learns which words are associated with positive (1) vs negative/neutral (0) sentiment.

In [9]:
# Create the model
model = LogisticRegression(
    max_iter=1000,   # Allow enough iterations for the model to converge
    random_state=42
)

# Train ("fit") the model on the training data
model.fit(X_train_tfidf, y_train)

print("Model trained successfully!")

Model trained successfully!


## Step 8: Evaluate the Model

We predict labels for the test set and compare them to the real labels.

**Understanding the Classification Report:**
| Metric | Meaning |
|---|---|
| **Precision** | Of all tweets predicted as class X, how many actually were X? |
| **Recall** | Of all actual class X tweets, how many did we correctly find? |
| **F1-score** | Harmonic mean of Precision & Recall (overall balance) |
| **Accuracy** | Overall % of tweets correctly classified |

In [10]:
# Generate predictions on the test set
y_pred = model.predict(X_test_tfidf)

# Print the full evaluation report
print("Classification Report")
print("=" * 50)
print(classification_report(
    y_test, y_pred,
    target_names=['Negative/Neutral (0)', 'Positive (1)']
))

Classification Report
                      precision    recall  f1-score   support

Negative/Neutral (0)       0.91      0.94      0.93    248563
        Positive (1)       0.88      0.82      0.85    121462

            accuracy                           0.90    370025
           macro avg       0.89      0.88      0.89    370025
        weighted avg       0.90      0.90      0.90    370025



## Step 9: Try It on a New Tweet!

Let's use our trained model to predict the sentiment of a brand-new tweet.

In [12]:
def predict_sentiment(tweet_text):
    """Takes a raw tweet string and returns the predicted sentiment."""
    # 1. Clean the tweet using our pipeline
    cleaned = text_cleaning_pipeline(tweet_text)
    # 2. Convert to TF-IDF vector (use the SAME vectorizer we already fitted)
    vector  = vectorizer.transform([cleaned])
    # 3. Predict
    pred    = model.predict(vector)[0]
    label   = "Positive" if pred == 1 else "Negative/Neutral"
    print(f"Tweet    : {tweet_text}")
    print(f"Cleaned  : {cleaned}")
    print(f"Sentiment: {label} (class {pred})")


# ── Try your own examples ──
predict_sentiment("Trump is doing a great job! America is winning!")
print()
predict_sentiment("This travel ban is unconstitutional and un-American.")

Tweet    : Trump is doing a great job! America is winning!
Cleaned  : trump great job america winning
Sentiment: Positive (class 1)

Tweet    : This travel ban is unconstitutional and un-American.
Cleaned  : travel ban unconstitutional unamerican
Sentiment: Negative/Neutral (class 0)
